# Day 1: Data audit (Tanzania maize leaf dataset)
Run the cells top to bottom. Dataset: https://data.mendeley.com/datasets/fkw49mz3xs/1 (DOI 10.17632/fkw49mz3xs.1). **Read the license on that page and note it down.**

## 1. Install and write the audit script

In [ ]:
!pip -q install pillow

In [ ]:
%%writefile day1_audit.py
"""Day 1 data audit for the Tanzania maize leaf dataset.

Usage:
    python day1_audit.py /path/to/extracted/dataset [--out audit_report.md]

Reports: class names and counts, image size stats, file naming patterns,
metadata-like files, exact duplicates (MD5), near duplicates (perceptual
hash), and exact duplicates that appear under different classes.
"""
import argparse
import collections
import hashlib
import os
import re
import sys

from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
META_EXT = {".csv", ".json", ".xml", ".txt", ".xlsx", ".xls", ".tsv", ".yaml", ".yml"}


def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    px = list(g.getdata())
    bits = 0
    for r in range(size):
        for c in range(size):
            bits = (bits << 1) | (px[r * (size + 1) + c] > px[r * (size + 1) + c + 1])
    return bits


def hamming(a, b):
    return bin(a ^ b).count("1")


def pattern(name):
    stem = os.path.splitext(name)[0]
    return re.sub(r"\d+", "#", stem)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("root")
    ap.add_argument("--out", default="audit_report.md")
    ap.add_argument("--near", type=int, default=4, help="max hamming distance for near-dup")
    a = ap.parse_args()

    images, metas, others = [], [], []
    for dp, _, fs in os.walk(a.root):
        for f in fs:
            p = os.path.join(dp, f)
            ext = os.path.splitext(f)[1].lower()
            if ext in IMG_EXT:
                images.append(p)
            elif ext in META_EXT:
                metas.append(p)
            else:
                others.append(p)

    if not images:
        sys.exit("No images found under " + a.root)

    def cls(p):
        rel = os.path.relpath(p, a.root).split(os.sep)
        return "/".join(rel[:-1]) if len(rel) > 1 else "(root)"

    per_class = collections.Counter(cls(p) for p in images)
    sizes = collections.Counter()
    modes = collections.Counter()
    patterns = collections.defaultdict(collections.Counter)
    bad = []
    hashes = collections.defaultdict(list)
    dh = []
    exif_gps = 0

    for i, p in enumerate(images, 1):
        try:
            with Image.open(p) as im:
                sizes[im.size] += 1
                modes[im.mode] += 1
                try:
                    ex = im.getexif()
                    if ex and 34853 in ex:  # GPSInfo tag
                        exif_gps += 1
                except Exception:
                    pass
                dh.append((dhash(im), p))
        except Exception as e:
            bad.append((p, str(e)))
            continue
        patterns[cls(p)][pattern(os.path.basename(p))] += 1
        hashes[md5(p)].append(p)
        if i % 500 == 0:
            print(f"  scanned {i}/{len(images)}", file=sys.stderr)

    exact = {h: ps for h, ps in hashes.items() if len(ps) > 1}
    cross = {h: ps for h, ps in exact.items() if len({cls(p) for p in ps}) > 1}

    # near duplicates: bucket by 16-bit prefix to avoid full O(n^2)
    buckets = collections.defaultdict(list)
    for h, p in dh:
        buckets[h >> 48].append((h, p))
    near = []
    for items in buckets.values():
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                d = hamming(items[i][0], items[j][0])
                if 0 < d <= a.near:
                    near.append((items[i][1], items[j][1], d))

    total = len(images)
    mx = max(per_class.values())
    mn = min(per_class.values())
    L = []
    L.append("# Day 1 audit report\n")
    L.append(f"Root: `{a.root}`\n")
    L.append(f"**Total images: {total}**  |  unreadable: {len(bad)}\n")
    L.append("## Class counts\n")
    L.append("| class | images | share |\n|---|---|---|")
    for c, n in sorted(per_class.items()):
        L.append(f"| {c} | {n} | {100*n/total:.1f}% |")
    L.append(f"\nImbalance ratio (largest/smallest): {mx/mn:.2f}\n")
    L.append("## Image sizes (top 10)\n")
    for s, n in sizes.most_common(10):
        L.append(f"- {s[0]}x{s[1]}: {n}")
    L.append(f"\nDistinct sizes: {len(sizes)}.  Modes: {dict(modes)}\n")
    L.append("## File naming patterns (top 3 per class)\n")
    for c, pc in sorted(patterns.items()):
        L.append(f"- {c}: " + "; ".join(f"`{p}` x{n}" for p, n in pc.most_common(3)))
    L.append("\n## Metadata-like files\n")
    if metas:
        for m in metas:
            L.append(f"- {os.path.relpath(m, a.root)}")
        L.append("\nOpen these and look for GPS, farm/plot IDs, dates, annotator fields.")
    else:
        L.append("None found. Farm/session split is not possible from files alone.")
    L.append(f"\nImages with EXIF GPS tag: {exif_gps} of {total}\n")
    if others:
        L.append("Other file types: " + ", ".join(sorted({os.path.splitext(o)[1].lower() or '(none)' for o in others})) + "\n")
    L.append("## Duplicates\n")
    L.append(f"- Exact duplicate groups: {len(exact)} ({sum(len(v)-1 for v in exact.values())} redundant files)")
    L.append(f"- Exact duplicates spanning different classes (label conflict): {len(cross)}")
    L.append(f"- Near-duplicate pairs (dHash distance <= {a.near}): {len(near)}")
    for p1, p2, d in near[:15]:
        L.append(f"  - d={d}: {os.path.relpath(p1, a.root)}  <->  {os.path.relpath(p2, a.root)}")
    if bad:
        L.append("\n## Unreadable files\n")
        for p, e in bad[:20]:
            L.append(f"- {p}: {e}")
    L.append("\n## Decision-point notes (fill in)\n")
    L.append("- Real total vs. 18,148 / 27,504: \n- Class balance workable?: \n- Linkable metadata?: \n- License: \n- Dup rate acceptable?: \n")

    with open(a.out, "w") as f:
        f.write("\n".join(L))
    print("\n".join(L))
    print(f"\nWritten to {a.out}", file=sys.stderr)


if __name__ == "__main__":
    main()


## 2. Get the data
Option A tries the Mendeley public download. If it fails (or the zip looks wrong), use Option B: download the zip by hand from the dataset page, upload it to Colab's file panel (or your Drive), and set `ZIP_PATH`.

In [ ]:
import os, subprocess
os.makedirs('/content/maize_zip', exist_ok=True)
ZIP_PATH = '/content/maize_zip/maize.zip'
url = 'https://data.mendeley.com/public-api/zip/fkw49mz3xs/download/1'
r = subprocess.run(['wget','-q','-O',ZIP_PATH,url])
print('wget exit code:', r.returncode)
!ls -lh /content/maize_zip
!file /content/maize_zip/maize.zip

In [ ]:
# Option B (only if A failed): upload the zip, or mount Drive and point ZIP_PATH at it.
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/<your-file>.zip'


## 3. Unzip (nested zips are expanded too)

In [ ]:
import zipfile, glob, os
DATA = '/content/maize_data'
os.makedirs(DATA, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA)
# expand any zips inside the zip
for _ in range(3):
    inner = [p for p in glob.glob(DATA + '/**/*.zip', recursive=True)]
    if not inner: break
    for p in inner:
        with zipfile.ZipFile(p) as z:
            z.extractall(p[:-4])
        os.remove(p)
!find /content/maize_data -maxdepth 3 -type d | head -40

## 4. Run the audit
If the report shows one huge `(root)` class or the wrong folder level, change `ROOT` to the folder that directly contains the class folders.

In [ ]:
ROOT = DATA
!python day1_audit.py "$ROOT" --out /content/audit_report.md

## 5. Save the report
Download it and paste it back into the chat for the decision-point write-up.

In [ ]:
from google.colab import files
files.download('/content/audit_report.md')